# π 幅度贝叶斯优化（真机入口）

目标函数为负 IQ 中心距离。输出只是候选脉冲参数；需要独立复测及门质量验证后再更新正式配置。用同样 shots/预算与网格扫描比较。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "execute":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in Platform_LQ or its execute directory")
sys.path.insert(0, str(ROOT))
import numpy as np
from src.pi import PulseSettings
from src.config import save_hardware
from optimize.runner import run_pi
BACKEND_NAME = "YOUR_BACKEND"
QUBIT = 9
BOUNDS = (0.5, 4.5)
STEPS = 21
SHOTS = 1000
KERNEL = "matern52"  # rbf, matern32, matern52, rq
ACQUISITION = "ei"   # ei or lcb
KAPPA = 2.0
SEED = 100
SETTINGS = PulseSettings()
RUN_HARDWARE = False
RUN_MODE = "new"
RESUME_RUN_TAG = ""

# Offline replay of the measured 21-point BO run; leave empty to use a new run.
PLOT_RUN_TAG = ""


In [ ]:
run = None
observations = None
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "pi", STEPS + 1,
                   run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG,
                   title='π 振幅贝叶斯优化'):
        from lqcloud import LQCloudProvider
        backend = LQCloudProvider().get_backend(BACKEND_NAME)
        qconf = {int(q["id"]): q for q in backend.config["properties"]["qubit_metrics"]}
        save_hardware(ROOT / "config", backend)
        run, observations = run_pi(
            backend, QUBIT, qconf[QUBIT], BOUNDS, ROOT / "result" / "pi",
            steps=STEPS, settings=SETTINGS, shots=SHOTS, kernel=KERNEL,
            acquisition=ACQUISITION, kappa=KAPPA, seed=SEED,
            run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG)
        best = max(observations, key=lambda row: row["S"])
        print("Run:", run)
        print("Candidate:", best["amplitude"], "IQ distance:", best["S"])


In [ ]:
# Load current or recorded measurements without submitting any QPU jobs.
import json
import pandas as pd
from IPython.display import display
from src.runs import load_manifest

if run is None and PLOT_RUN_TAG:
    if (Path(PLOT_RUN_TAG).name != PLOT_RUN_TAG
            or "/" in PLOT_RUN_TAG or "\\" in PLOT_RUN_TAG):
        raise ValueError("PLOT_RUN_TAG must be one result directory name")
    run = ROOT / "result" / "pi" / PLOT_RUN_TAG
if run is not None:
    manifest = load_manifest(run)
    if manifest.get("kind") != "pi" or manifest.get("strategy") != "bo":
        raise ValueError("Selected run is not a pi-amplitude BO run")
    csv_path = run / "data" / "observations.csv"
    data = pd.read_csv(csv_path)
    required = {"amplitude", "loss", "noise_variance", "S", "mean_displacement_noise"}
    if not required.issubset(data.columns) or data.empty:
        raise ValueError("BO observations are missing or invalid")
    data = data.astype({key: float for key in required})
    if not np.isfinite(data[list(required)].to_numpy()).all():
        raise ValueError("BO observations contain nonfinite values")
    bounds = tuple(manifest["plan"]["bounds"])
    kernel = manifest["plan"]["kernel"]
    seed = int(manifest["plan"]["seed"])
    initial_points = int(manifest["plan"]["initial_points"])
    print(f"Loaded {len(data)} points from {run.name} ({manifest['status']})")
    display(data[["amplitude", "S", "mean_displacement_noise", "loss"]])
else:
    print("No data: set PLOT_RUN_TAG to a saved run or enable RUN_HARDWARE.")


## 贝叶斯优化的函数图像与过程

蓝线为 GP 对目标函数 L(A)=-S(A) 的后验拟合（图上绘制 S(A)，数值越高越好）；阴影表示模型不确定性。红星是当前最新测量点，绿线是截至该轮的最佳实测振幅。最后附上条件性逐点留一预测、最佳实测值的变化；这些诊断不等同于独立实验验证或门保真度。


In [ ]:
# The shared plotter replays saved BO observations and displays both figures.
from optimize.show_fig import plot_bo_run

if run is not None:
    figures = plot_bo_run(run)
